# 🎨 [컴비 2조] 가짜 생성 C (SDXL) - 20종 물건별 맞춤 제작 스튜디오
**실행 방법:** 상단 메뉴 `런타임` > `런타임 유형 변경` > **T4 GPU** 선택 후 실행하세요.


### 📌 [1번 셀] 필수 라이브러리 설치 및 SDXL 모델 로드


In [ ]:
!pip install -q diffusers transformers accelerate safetensors

import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"

# fp16 충돌 방지 전용 VAE 로드
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16
)

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
).to("cuda")
print("✅ SDXL 1.0 모델 로드 완료!")

### 📌 [2번 셀] 핵심 프롬프트 템플릿 및 generate() 함수 정의


In [ ]:
import os, csv, random, shutil
import torch
from datetime import datetime
from IPython.display import display

OUT_DIR = "fakeC"
os.makedirs(OUT_DIR, exist_ok=True)
LOG_PATH = f"{OUT_DIR}/log.csv"

# 토큰 길이 검증 함수 (75토큰 이하여야 뒤가 안 잘림)
def tok_len(s):
    return len(pipe.tokenizer(s).input_ids) - 2

# 팀 최적화 초경량 네거티브 베이스 (39토큰으로 압축하여 77토큰 잘림 완벽 방지)
NEGATIVE_BASE = (
    "multiple sticky notes, multiple papers, glowing screen, text on screen, cartoon, "
    "3d render, gibberish, deformed, watermark, logo, cable, close-up, cropped"
)

# 기본 프롬프트 템플릿
TEMPLATE = (
    'a casual snapshot of a {obj} placed on a {bg}, '
    'with only one {paper} beside it with handwritten date "{date}", '
    '{angle}, {light}, realistic, authentic amateur photo, clean shot'
)

def generate(obj, bg, light, paper="small yellow sticky note", angle="angled 45-degree view", date="2026.11.18", seed=None, idx=0, steps=30, guidance=7.5, extra_negative="", record_log=True):
    seed = seed if seed is not None else random.randint(0, 2**31 - 1)
    prompt = TEMPLATE.format(obj=obj, bg=bg, light=light, paper=paper, angle=angle, date=date)
    neg_prompt = f"{NEGATIVE_BASE}, {extra_negative}".strip(", ")
    p_len = tok_len(prompt)
    n_len = tok_len(neg_prompt)
    print(f"토큰 수 → positive {p_len} / negative {n_len} (75 이하여야 함)")
    if p_len > 75:
        print(f"⚠️ 경고: 프롬프트가 {p_len}토큰으로 75토큰을 초과했습니다. 뒤쪽 단어가 잘릴 수 있습니다.")
    if n_len > 75:
        print(f"⚠️ 경고: 네거티브가 {n_len}토큰으로 75토큰을 초과했습니다. 뒤쪽 단어가 잘릴 수 있습니다.")

    g = torch.Generator("cuda").manual_seed(seed)

    img = pipe(
        prompt,
        negative_prompt=neg_prompt,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=g
    ).images[0]

    fname = f"fakeC_{idx:04d}.png"
    img.save(f"{OUT_DIR}/{fname}")

    if record_log:
        new = not os.path.exists(LOG_PATH)
        with open(LOG_PATH, "a", newline="", encoding="utf-8-sig") as f:
            w = csv.writer(f)
            if new:
                w.writerow(["파일명", "유형", "품목", "프롬프트", "negative", "seed", "모델", "steps", "guidance", "생성시각"])
            w.writerow([fname, "C", obj, prompt, neg_prompt, seed, "SDXL-base-1.0", steps, guidance, datetime.now().isoformat()])

    return img
print("✅ 팀 최적화 39토큰 네거티브 & 토큰 길이 검증기(tok_len) 장착 완료!")

## 🔍 [3번 섹션] 20종 물건별 구도 확인 (3-1 ~ 3-20 연속)
먼저 3-1부터 3-20까지 3장씩 돌려 구도를 확인하세요. (추가 4종 확정 시 3-21 ~ 3-24 추가 예정)


#### 📌 [3-1] 휴대폰 (smartphone) 구도 확인 (3장)


In [ ]:
# [3-1] 휴대폰 (smartphone) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='black smartphone lying flat, screen off',
        bg='light wooden desk',
        light='soft window light',
        paper='small yellow sticky note',
        angle='top-down view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9010 + i,   # fakeC_9010 ~ 9012
        steps=35,
        guidance=7.0,
        extra_negative='pencil, pen, bowl, cloth, feature phone, keypad, two phones',
    )
    print(f"[{i+1}/3] 휴대폰 완료")
    display(img)

#### 📌 [3-2] 노트북 (laptop) 구도 확인 (3장)


In [ ]:
# [3-2] 노트북 (laptop) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='open laptop with black blank screen',
        bg='wooden study desk',
        light='natural window sunlight',
        paper='small white memo paper beside the keyboard',
        angle='angled 45-degree top-down view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9020 + i,   # fakeC_9020 ~ 9022
        steps=35,
        guidance=7.0,
        extra_negative='two laptops',
    )
    print(f"[{i+1}/3] 노트북 완료")
    display(img)

#### 📌 [3-3] 워치 (smartwatch) 구도 확인 (3장)


In [ ]:
# [3-3] 워치 (smartwatch) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='smartwatch with black strap, screen off',
        bg='wooden nightstand table',
        light='bright indoor fluorescent light',
        paper='small yellow sticky note',
        angle='overhead top-down flat lay shot',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9030 + i,   # fakeC_9030 ~ 9032
        steps=35,
        guidance=7.0,
        extra_negative='two watches',
    )
    print(f"[{i+1}/3] 워치 완료")
    display(img)

#### 📌 [3-4] 이어폰 (earbuds) 구도 확인 (3장)


In [ ]:
# [3-4] 이어폰 (earbuds) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='wireless earbuds charging case',
        bg='white study desk',
        light='bright indoor fluorescent light',
        paper='small paper receipt',
        angle='angled 45-degree top-down view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9040 + i,   # fakeC_9040 ~ 9042
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 이어폰 완료")
    display(img)

#### 📌 [3-5] 헤드셋 (headphones) 구도 확인 (3장)


In [ ]:
# [3-5] 헤드셋 (headphones) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='over-ear headphones',
        bg='wooden study desk',
        light='dim warm desk lamp lighting',
        paper='small yellow sticky note beside earcups',
        angle='angled 45-degree top-down view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9050 + i,   # fakeC_9050 ~ 9052
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 헤드셋 완료")
    display(img)

#### 📌 [3-6] 키보드 (keyboard) 구도 확인 (3장)


In [ ]:
# [3-6] 키보드 (keyboard) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='mechanical keyboard on a desk mat',
        bg='wooden study desk',
        light='natural window sunlight',
        paper='small yellow sticky note in front of spacebar',
        angle='angled 45-degree top-down view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9060 + i,   # fakeC_9060 ~ 9062
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 키보드 완료")
    display(img)

#### 📌 [3-7] 마우스 (mouse) 구도 확인 (3장)


In [ ]:
# [3-7] 마우스 (mouse) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='wireless mouse on a mousepad',
        bg='wooden desk',
        light='natural window sunlight',
        paper='small yellow sticky note beside mouse',
        angle='angled 45-degree view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9070 + i,   # fakeC_9070 ~ 9072
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 마우스 완료")
    display(img)

#### 📌 [3-8] 신발 (sneakers) 구도 확인 (3장)


In [ ]:
# [3-8] 신발 (sneakers) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='pair of sneakers on top of a shoebox',
        bg='hardwood floor',
        light='natural window sunlight',
        paper='small white memo paper beside shoes',
        angle='angled 45-degree top-down view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9080 + i,   # fakeC_9080 ~ 9082
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 신발 완료")
    display(img)

#### 📌 [3-9] 텀블러 (tumbler) 구도 확인 (3장)


In [ ]:
# [3-9] 텀블러 (tumbler) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='stainless steel tumbler',
        bg='wooden study desk',
        light='bright indoor fluorescent light',
        paper='small paper receipt beside tumbler base',
        angle='eye-level front view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9090 + i,   # fakeC_9090 ~ 9092
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 텀블러 완료")
    display(img)

#### 📌 [3-10] 옷 (hoodie) 구도 확인 (3장)


In [ ]:
# [3-10] 옷 (hoodie) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='neatly folded hoodie',
        bg='wrinkled bed sheet',
        light='bright indoor fluorescent light',
        paper='small yellow sticky note placed on the hoodie',
        angle='overhead top-down flat lay shot',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9100 + i,   # fakeC_9100 ~ 9102
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 옷 완료")
    display(img)

#### 📌 [3-11] 의자 (chair) 구도 확인 (3장)


In [ ]:
# [3-11] 의자 (chair) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='ergonomic mesh desk chair',
        bg='living room wooden floor',
        light='bright indoor fluorescent light',
        paper='small yellow sticky note placed on the chair seat cushion',
        angle='eye-level front 45-degree view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9110 + i,   # fakeC_9110 ~ 9112
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 의자 완료")
    display(img)

#### 📌 [3-12] 지갑 (wallet) 구도 확인 (3장)


In [ ]:
# [3-12] 지갑 (wallet) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='leather wallet',
        bg='wooden table',
        light='dim warm desk lamp lighting',
        paper='small paper receipt beside wallet',
        angle='overhead top-down flat lay shot',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9120 + i,   # fakeC_9120 ~ 9122
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 지갑 완료")
    display(img)

#### 📌 [3-13] 자동차 (car) 구도 확인 (3장)


In [ ]:
# [3-13] 자동차 (car) 구도 확인 (3장)
for i in range(3):
    # (자동차 실내 핸들 구도로 바꾸려면 obj="car steering wheel and dashboard interior" 로 변경)
    img = generate(
        obj='parked sedan car hood and front windshield',
        bg='outdoor parking lot',
        light='natural daylight',
        paper='small yellow sticky note stuck on the car hood in foreground',
        angle='angled 45-degree view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9130 + i,   # fakeC_9130 ~ 9132
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 자동차 완료")
    display(img)

#### 📌 [3-14] 가방 (backpack) 구도 확인 (3장)


In [ ]:
# [3-14] 가방 (backpack) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='everyday backpack sitting upright',
        bg='clean floor against wall',
        light='natural window sunlight',
        paper='small yellow sticky note attached to the front pocket',
        angle='eye-level front 45-degree view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9140 + i,   # fakeC_9140 ~ 9142
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 가방 완료")
    display(img)

#### 📌 [3-15] 책 (book) 구도 확인 (3장)


In [ ]:
# [3-15] 책 (book) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='paperback book',
        bg='wooden study desk',
        light='bright indoor fluorescent light',
        paper='small yellow sticky note on the book cover corner',
        angle='angled 45-degree top-down view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9150 + i,   # fakeC_9150 ~ 9152
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 책 완료")
    display(img)

#### 📌 [3-16] 화장품 (cosmetics) 구도 확인 (3장)


In [ ]:
# [3-16] 화장품 (cosmetics) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='cosmetics serum bottle',
        bg='white vanity table',
        light='natural window sunlight',
        paper='small white memo paper beside bottle',
        angle='angled 45-degree top-down view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9160 + i,   # fakeC_9160 ~ 9162
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 화장품 완료")
    display(img)

#### 📌 [3-17] 냉장고 (fridge) 구도 확인 (3장)


In [ ]:
# [3-17] 냉장고 (fridge) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='compact mini fridge in room corner',
        bg='wooden floor',
        light='natural window sunlight',
        paper='small yellow sticky note on the mini fridge door',
        angle='eye-level front 45-degree view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9170 + i,   # fakeC_9170 ~ 9172
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 냉장고 완료")
    display(img)

#### 📌 [3-18] 삼각대 (tripod) 구도 확인 (3장)


In [ ]:
# [3-18] 삼각대 (tripod) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='standing camera tripod',
        bg='wooden living room floor',
        light='bright indoor fluorescent light',
        paper='small white memo paper placed beside tripod leg base',
        angle='eye-level view shot',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9180 + i,   # fakeC_9180 ~ 9182
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 삼각대 완료")
    display(img)

#### 📌 [3-19] 매트리스 (mattress) 구도 확인 (3장)


In [ ]:
# [3-19] 매트리스 (mattress) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='mattress on the floor with bedsheet',
        bg='bedroom floor',
        light='natural window sunlight',
        paper='small yellow sticky note on the corner edge of the mattress',
        angle='angled 45-degree top-down view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9190 + i,   # fakeC_9190 ~ 9192
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 매트리스 완료")
    display(img)

#### 📌 [3-20] 책상 (desk) 구도 확인 (3장)


In [ ]:
# [3-20] 책상 (desk) 구도 확인 (3장)
for i in range(3):
    img = generate(
        obj='wooden study desk in a room',
        bg='room floor against wall',
        light='dim warm desk lamp lighting',
        paper='small yellow sticky note on the center of the desk surface',
        angle='eye-level front 45-degree view',
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=9200 + i,   # fakeC_9200 ~ 9202
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 책상 완료")
    display(img)

## 🚀 [4번 섹션] 20종 물건별 15장 본 양산 (4-1 ~ 4-20 연속)
3-1 ~ 3-20 구도 확인이 끝난 뒤, 아래 4-1 ~ 4-20을 돌려 물건당 15장씩 생성하세요.


#### 🚀 [4-1] 휴대폰 (smartphone) 15장 양산


In [ ]:
# [4-1] 휴대폰 (smartphone) 15장 양산 (3-1 설정 기반)
COUNT = 15
IDX_START = 101   # 파일 번호: fakeC_0101 ~ 0115

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small yellow sticky note'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='black smartphone lying flat, screen off',
        bg='light wooden desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='pencil, pen, bowl, cloth, feature phone, keypad, two phones',
    )
    print(f"[{i+1}/{COUNT}] 휴대폰 완료")
    display(img)
print("🎉 [휴대폰] 15장 완료!")

#### 🚀 [4-2] 노트북 (laptop) 15장 양산


In [ ]:
# [4-2] 노트북 (laptop) 15장 양산 (3-2 설정 기반)
COUNT = 15
IDX_START = 201   # 파일 번호: fakeC_0201 ~ 0215

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small white memo paper beside the keyboard'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='open laptop with black blank screen',
        bg='wooden study desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='two laptops',
    )
    print(f"[{i+1}/{COUNT}] 노트북 완료")
    display(img)
print("🎉 [노트북] 15장 완료!")

#### 🚀 [4-3] 워치 (smartwatch) 15장 양산


In [ ]:
# [4-3] 워치 (smartwatch) 15장 양산 (3-3 설정 기반)
COUNT = 15
IDX_START = 301   # 파일 번호: fakeC_0301 ~ 0315

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small yellow sticky note'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='smartwatch with black strap, screen off',
        bg='wooden nightstand table',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='two watches',
    )
    print(f"[{i+1}/{COUNT}] 워치 완료")
    display(img)
print("🎉 [워치] 15장 완료!")

#### 🚀 [4-4] 이어폰 (earbuds) 15장 양산


In [ ]:
# [4-4] 이어폰 (earbuds) 15장 양산 (3-4 설정 기반)
COUNT = 15
IDX_START = 401   # 파일 번호: fakeC_0401 ~ 0415

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small paper receipt'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='wireless earbuds charging case',
        bg='white study desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 이어폰 완료")
    display(img)
print("🎉 [이어폰] 15장 완료!")

#### 🚀 [4-5] 헤드셋 (headphones) 15장 양산


In [ ]:
# [4-5] 헤드셋 (headphones) 15장 양산 (3-5 설정 기반)
COUNT = 15
IDX_START = 501   # 파일 번호: fakeC_0501 ~ 0515

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small yellow sticky note beside earcups'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='over-ear headphones',
        bg='wooden study desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 헤드셋 완료")
    display(img)
print("🎉 [헤드셋] 15장 완료!")

#### 🚀 [4-6] 키보드 (keyboard) 15장 양산


In [ ]:
# [4-6] 키보드 (keyboard) 15장 양산 (3-6 설정 기반)
COUNT = 15
IDX_START = 601   # 파일 번호: fakeC_0601 ~ 0615

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small yellow sticky note in front of spacebar'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='mechanical keyboard on a desk mat',
        bg='wooden study desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 키보드 완료")
    display(img)
print("🎉 [키보드] 15장 완료!")

#### 🚀 [4-7] 마우스 (mouse) 15장 양산


In [ ]:
# [4-7] 마우스 (mouse) 15장 양산 (3-7 설정 기반)
COUNT = 15
IDX_START = 701   # 파일 번호: fakeC_0701 ~ 0715

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small yellow sticky note beside mouse'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='wireless mouse on a mousepad',
        bg='wooden desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 마우스 완료")
    display(img)
print("🎉 [마우스] 15장 완료!")

#### 🚀 [4-8] 신발 (sneakers) 15장 양산


In [ ]:
# [4-8] 신발 (sneakers) 15장 양산 (3-8 설정 기반)
COUNT = 15
IDX_START = 801   # 파일 번호: fakeC_0801 ~ 0815

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['angled 45-degree view', 'eye-level front view', 'eye-level side view']
PAPER_BASE = 'small white memo paper beside shoes'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='pair of sneakers on top of a shoebox',
        bg='hardwood floor',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 신발 완료")
    display(img)
print("🎉 [신발] 15장 완료!")

#### 🚀 [4-9] 텀블러 (tumbler) 15장 양산


In [ ]:
# [4-9] 텀블러 (tumbler) 15장 양산 (3-9 설정 기반)
COUNT = 15
IDX_START = 901   # 파일 번호: fakeC_0901 ~ 0915

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small paper receipt beside tumbler base'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='stainless steel tumbler',
        bg='wooden study desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 텀블러 완료")
    display(img)
print("🎉 [텀블러] 15장 완료!")

#### 🚀 [4-10] 옷 (hoodie) 15장 양산


In [ ]:
# [4-10] 옷 (hoodie) 15장 양산 (3-10 설정 기반)
COUNT = 15
IDX_START = 1001   # 파일 번호: fakeC_1001 ~ 1015

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small yellow sticky note placed on the hoodie'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='neatly folded hoodie',
        bg='wrinkled bed sheet',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 옷 완료")
    display(img)
print("🎉 [옷] 15장 완료!")

#### 🚀 [4-11] 의자 (chair) 15장 양산


In [ ]:
# [4-11] 의자 (chair) 15장 양산 (3-11 설정 기반)
COUNT = 15
IDX_START = 1101   # 파일 번호: fakeC_1101 ~ 1115

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['angled 45-degree view', 'eye-level front view', 'eye-level side view']
PAPER_BASE = 'small yellow sticky note placed on the chair seat cushion'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='ergonomic mesh desk chair',
        bg='living room wooden floor',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 의자 완료")
    display(img)
print("🎉 [의자] 15장 완료!")

#### 🚀 [4-12] 지갑 (wallet) 15장 양산


In [ ]:
# [4-12] 지갑 (wallet) 15장 양산 (3-12 설정 기반)
COUNT = 15
IDX_START = 1201   # 파일 번호: fakeC_1201 ~ 1215

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small paper receipt beside wallet'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='leather wallet',
        bg='wooden table',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 지갑 완료")
    display(img)
print("🎉 [지갑] 15장 완료!")

#### 🚀 [4-13] 자동차 (car) 15장 양산


In [ ]:
# [4-13] 자동차 (car) 15장 양산 (3-13 설정 기반)
COUNT = 15
IDX_START = 1301   # 파일 번호: fakeC_1301 ~ 1315

lights = ['natural daylight', 'cloudy daylight', 'evening outdoor light']
angles = ['angled 45-degree view', 'eye-level front view', 'eye-level side view']
PAPER_BASE = 'small yellow sticky note stuck on the car hood in foreground'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='parked sedan car hood and front windshield',
        bg='outdoor parking lot',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 자동차 완료")
    display(img)
print("🎉 [자동차] 15장 완료!")

#### 🚀 [4-14] 가방 (backpack) 15장 양산


In [ ]:
# [4-14] 가방 (backpack) 15장 양산 (3-14 설정 기반)
COUNT = 15
IDX_START = 1401   # 파일 번호: fakeC_1401 ~ 1415

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['angled 45-degree view', 'eye-level front view', 'eye-level side view']
PAPER_BASE = 'small yellow sticky note attached to the front pocket'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='everyday backpack sitting upright',
        bg='clean floor against wall',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 가방 완료")
    display(img)
print("🎉 [가방] 15장 완료!")

#### 🚀 [4-15] 책 (book) 15장 양산


In [ ]:
# [4-15] 책 (book) 15장 양산 (3-15 설정 기반)
COUNT = 15
IDX_START = 1501   # 파일 번호: fakeC_1501 ~ 1515

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small yellow sticky note on the book cover corner'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='paperback book',
        bg='wooden study desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 책 완료")
    display(img)
print("🎉 [책] 15장 완료!")

#### 🚀 [4-16] 화장품 (cosmetics) 15장 양산


In [ ]:
# [4-16] 화장품 (cosmetics) 15장 양산 (3-16 설정 기반)
COUNT = 15
IDX_START = 1601   # 파일 번호: fakeC_1601 ~ 1615

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small white memo paper beside bottle'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='cosmetics serum bottle',
        bg='white vanity table',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 화장품 완료")
    display(img)
print("🎉 [화장품] 15장 완료!")

#### 🚀 [4-17] 냉장고 (fridge) 15장 양산


In [ ]:
# [4-17] 냉장고 (fridge) 15장 양산 (3-17 설정 기반)
COUNT = 15
IDX_START = 1701   # 파일 번호: fakeC_1701 ~ 1715

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['angled 45-degree view', 'eye-level front view', 'eye-level side view']
PAPER_BASE = 'small yellow sticky note on the mini fridge door'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='compact mini fridge in room corner',
        bg='wooden floor',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 냉장고 완료")
    display(img)
print("🎉 [냉장고] 15장 완료!")

#### 🚀 [4-18] 삼각대 (tripod) 15장 양산


In [ ]:
# [4-18] 삼각대 (tripod) 15장 양산 (3-18 설정 기반)
COUNT = 15
IDX_START = 1801   # 파일 번호: fakeC_1801 ~ 1815

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['angled 45-degree view', 'eye-level front view', 'eye-level side view']
PAPER_BASE = 'small white memo paper placed beside tripod leg base'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='standing camera tripod',
        bg='wooden living room floor',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 삼각대 완료")
    display(img)
print("🎉 [삼각대] 15장 완료!")

#### 🚀 [4-19] 매트리스 (mattress) 15장 양산


In [ ]:
# [4-19] 매트리스 (mattress) 15장 양산 (3-19 설정 기반)
COUNT = 15
IDX_START = 1901   # 파일 번호: fakeC_1901 ~ 1915

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['angled 45-degree view', 'eye-level front view', 'eye-level side view']
PAPER_BASE = 'small yellow sticky note on the corner edge of the mattress'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='mattress on the floor with bedsheet',
        bg='bedroom floor',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 매트리스 완료")
    display(img)
print("🎉 [매트리스] 15장 완료!")

#### 🚀 [4-20] 책상 (desk) 15장 양산


In [ ]:
# [4-20] 책상 (desk) 15장 양산 (3-20 설정 기반)
COUNT = 15
IDX_START = 2001   # 파일 번호: fakeC_2001 ~ 2015

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['angled 45-degree view', 'eye-level front view', 'eye-level side view']
PAPER_BASE = 'small yellow sticky note on the center of the desk surface'

for i in range(COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='wooden study desk in a room',
        bg='room floor against wall',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date='2026-11-18',
        seed=random.randint(1, 99999),
        idx=IDX_START + i,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/{COUNT}] 책상 완료")
    display(img)
print("🎉 [책상] 15장 완료!")

### 📌 [5번 셀] 생성 결과 압축 및 다운로드 (`fakeC.zip`)


In [ ]:
!zip -rq fakeC.zip "{OUT_DIR}"
from google.colab import files
files.download("fakeC.zip")
print("📦 다운로드 완료!")